In [13]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [14]:
import os
import cv2
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import spacy

from PIL import Image

import torchvision.transforms as transforms

In [2]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
nlp = spacy.load("en_core_web_sm")

In [16]:
dataset_path = r"C:\Users\Lenovo\Internship\data\Flickr8k"
image_folder = os.path.join(dataset_path, "images")
caption_file = os.path.join(dataset_path, "captions.txt")

In [17]:
captions = pd.read_csv(caption_file, sep="|")

captions.head()

,image_name,caption_number,caption_text
0,1000268201_693b08cb0e.jpg,0,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,1,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,2,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,3,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,4,A little girl in a pink dress going into a woo...


In [18]:
print(captions.columns)

Index(['image_name', 'caption_number', 'caption_text'], dtype='str')


In [19]:
captions["clean_caption"] = captions["caption_text"].str.lower()

captions[["caption_text", "clean_caption"]].head()

,caption_text,clean_caption
0,A child in a pink dress is climbing up a set o...,a child in a pink dress is climbing up a set o...
1,A girl going into a wooden building .,a girl going into a wooden building .
2,A little girl climbing into a wooden playhouse .,a little girl climbing into a wooden playhouse .
3,A little girl climbing the stairs to her playh...,a little girl climbing the stairs to her playh...
4,A little girl in a pink dress going into a woo...,a little girl in a pink dress going into a woo...


In [20]:
print(captions.columns)
captions.head()

Index(['image_name', 'caption_number', 'caption_text', 'clean_caption'], dtype='str')


,image_name,caption_number,caption_text,clean_caption
0,1000268201_693b08cb0e.jpg,0,A child in a pink dress is climbing up a set o...,a child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,1,A girl going into a wooden building .,a girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,2,A little girl climbing into a wooden playhouse .,a little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,3,A little girl climbing the stairs to her playh...,a little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,4,A little girl in a pink dress going into a woo...,a little girl in a pink dress going into a woo...


In [21]:
captions["tokens"] = captions["clean_caption"].apply(word_tokenize)

captions[["caption_text", "tokens"]].head()

,caption_text,tokens
0,A child in a pink dress is climbing up a set o...,"[a, child, in, a, pink, dress, is, climbing, u..."
1,A girl going into a wooden building .,"[a, girl, going, into, a, wooden, building, .]"
2,A little girl climbing into a wooden playhouse .,"[a, little, girl, climbing, into, a, wooden, p..."
3,A little girl climbing the stairs to her playh...,"[a, little, girl, climbing, the, stairs, to, h..."
4,A little girl in a pink dress going into a woo...,"[a, little, girl, in, a, pink, dress, going, i..."


In [22]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

captions["filtered_tokens"] = captions["tokens"].apply(
    lambda words: [word for word in words if word.isalpha() and word not in stop_words]
)

captions[["caption_text", "filtered_tokens"]].head()

,caption_text,filtered_tokens
0,A child in a pink dress is climbing up a set o...,"[child, pink, dress, climbing, set, stairs, en..."
1,A girl going into a wooden building .,"[girl, going, wooden, building]"
2,A little girl climbing into a wooden playhouse .,"[little, girl, climbing, wooden, playhouse]"
3,A little girl climbing the stairs to her playh...,"[little, girl, climbing, stairs, playhouse]"
4,A little girl in a pink dress going into a woo...,"[little, girl, pink, dress, going, wooden, cabin]"


In [23]:
vocab = sorted(set(
    word
    for tokens in captions["filtered_tokens"]
    for word in tokens
))

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 8254


In [24]:
processed_folder = r"C:\Users\Lenovo\Internship\data\processed"

import os
os.makedirs(processed_folder, exist_ok=True)

print("Folder created successfully!")

Folder created successfully!


In [25]:
captions.to_csv(
    os.path.join(processed_folder, "processed_captions.csv"),
    index=False
)

print("Processed captions saved successfully!")

Processed captions saved successfully!


In [26]:
processed_images = os.path.join(processed_folder, "images")

os.makedirs(processed_images, exist_ok=True)

print("Processed image folder created!")

Processed image folder created!


In [27]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [28]:
image_list = captions["image_name"].unique()

print("Total Images:", len(image_list))

Total Images: 8091


In [29]:
from PIL import Image

for img_name in image_list:

    img_path = os.path.join(image_folder, img_name)

    image = cv2.imread(img_path)

    if image is None:
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    tensor = transform(image)

    save_img = transforms.ToPILImage()(tensor)

    save_img.save(os.path.join(processed_images, img_name))

print("All images processed successfully!")

All images processed successfully!


In [30]:
for img_name in image_list[:5]:

    img = cv2.imread(os.path.join(image_folder, img_name))

    print(img_name, img.shape)

1000268201_693b08cb0e.jpg (500, 375, 3)
1001773457_577c3a7d70.jpg (375, 500, 3)
1002674143_1b742ab4b8.jpg (400, 500, 3)
1003163366_44323f5815.jpg (410, 500, 3)
1007129816_e794419615.jpg (461, 500, 3)


In [31]:
image = cv2.imread(os.path.join(image_folder, image_list[0]))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

tensor = transform(image)

print("Minimum Pixel Value:", tensor.min().item())
print("Maximum Pixel Value:", tensor.max().item())

Minimum Pixel Value: 0.003921568859368563
Maximum Pixel Value: 1.0
